# 3 · Train, evaluate and export the models

Uses the search result from notebook 2 (`data/models/hyperparameters.json`, or defaults if it does not exist yet) and

1. evaluates each model **leave-one-sausage-out**: train on four sausages, score on the fifth;
2. saves one model per fold (`data/models/folds/`). The demonstrator replays a sausage with the model that never saw it;
3. trains the final models on all annotated sausages (`data/models/final/`);
4. writes `manifest.json` and `metrics.json`, which the demonstrator's evaluation panel shows.

`LIMIT_SIZE` caps the Random Forest (100 trees, depth 10). An unlimited forest on this data grows to hundreds of MB.
**Train with the same scikit-learn version the demo laptop runs**, because model files are not portable across versions.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, so ml_core is importable
import json
import pandas as pd
from ml_core import data
from ml_core.training import load_windows, metrics_table, train_and_evaluate

In [ ]:
CONVERTED_DIR = data.CONVERTED_DIR
MODELS_DIR = data.MODELS_DIR
WINDOW_SIZE, STRIDE = 10, 2
LIMIT_SIZE = True
FOLD_MODELS = True
QUICK = False

In [ ]:
windows = load_windows(CONVERTED_DIR, window_size=WINDOW_SIZE, stride=STRIDE)
windows.summary()

In [ ]:
metrics = train_and_evaluate(windows, MODELS_DIR, limit_size=LIMIT_SIZE, fold_models=FOLD_MODELS, quick=QUICK)

## Results (mean ± std over the held-out sausages)

In [ ]:
metrics_table(metrics)

In [ ]:
# balanced accuracy per held-out sausage
pd.DataFrame({m["name"]: {f["held_out"]: round(f["balanced_accuracy"], 3) for f in m["folds"]}
              for m in metrics["models"].values()})

In [ ]:
# confusion matrices, row-normalised (rows: annotation, columns: prediction)
for m in metrics["models"].values():
    cm = pd.DataFrame(m["confusion_matrix"], index=metrics["classes"], columns=metrics["classes"])
    print(m["name"])
    display((cm.div(cm.sum(axis=1), axis=0) * 100).round(1))

In [ ]:
# how often two models give the same label on held-out windows
pd.Series(metrics["agreement_on_held_out"]).round(3)

**Next:** start the demonstrator (`scripts/start_demo.ps1`). It loads the models from `data/models/`.
Read the scores with care: all annotated sausages come from one curing run.